# Data Cleaning & Preprocessing

Notebook ini didedikasikan untuk melakukan ekstraksi, transformasi, dan pembersihan (*cleaning*) dataset UMKM. Hasil dari notebook ini adalah dataset yang sudah bersih dan kaya akan fitur numerik yang siap digunakan untuk berbagai kasus klastering (Case 1 - 7).

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings('ignore')

### 1. Load Data Mentah

In [2]:
# Asumsi dataset berada di folder data
df_raw = pd.read_excel('../data/Data Set UMKM.xlsx', header=1)
print(f"Total baris data mentah: {len(df_raw)}")
df_raw.head(3)

Total baris data mentah: 2769


,Usia,Jenis Kelamin,Pendidikan Terakhir,Provinsi,Kab/Kota,Kecamatan,"Desa/Kel, RT, RW",Nama Jalan,Nama Usaha,Tanggal Pendirian Usaha,...,Status Kepemilkan Tanah/Bangunan,Sarana Media Elektronik,Modal Bantuan Pemerintah,Pinjaman Kredit Usaha Rakyat,Omset per-Tahun,Kepemilikan Asuransi Kesehatan,Laki-laki,Perempuan,Rerata Usia Pekerja,Status Formulir
0,42,P,-,DAERAH ISTIMEWA YOGYAKARTA,KOTA YOGYAKARTA,NGAMPILAN,"NGAMPILAN, 27, 5",SANGGRAHAN PATHUK NG I/534,GAYATRI ECOPRINT BATIK JUMPUTAN SHIBORI DAN CRAFT,-,...,NaN,-,-,-,-,-,-,-,-,Terverifikasi
1,41,P,SMA,DAERAH ISTIMEWA YOGYAKARTA,KOTA YOGYAKARTA,TEGALREJO,"KARANGAWARU, 21, 6",BLUNYAHREJO TR.II/839,NASYWA SNACK,24 Juli 2012,...,Lainnya,"WhatsApp, Facebook",-,-,Kurang dari 10 juta,-,0,0,-,Terverifikasi
2,41,P,D3,DAERAH ISTIMEWA YOGYAKARTA,KOTA YOGYAKARTA,DANUREJAN,"TEGALPANGGUNG, 2, 1",LEDOK TUKANGAN DN.2/231 B,WARUNG ABINAYA,10 Agustus 2020,...,Milik sendiri,"WhatsApp, Facebook, Instagram",Pemkot Yogyakarta,-,Kurang dari 10 juta,-,0,0,35-50 tahun,Terverifikasi


### 2. Penanganan Missing Values dan Tipe Data
Beberapa kolom numerik seperti usia dan jumlah pekerja terkadang diisi dengan strip ('-'). Kita akan mengubahnya menjadi angka 0 atau rata-rata.

In [3]:
df = df_raw.copy()

# Mengubah strip '-' menjadi 0 untuk kolom numerik dasar
cols_to_zero = ['Usia', 'Laki-laki', 'Perempuan']
df[cols_to_zero] = df[cols_to_zero].replace('-', 0).fillna(0)

# Casting tipe data
df['Usia'] = pd.to_numeric(df['Usia'], errors='coerce').fillna(0)
df['Laki-laki'] = pd.to_numeric(df['Laki-laki'], errors='coerce').fillna(0)
df['Perempuan'] = pd.to_numeric(df['Perempuan'], errors='coerce').fillna(0)

# Drop baris yang tidak memiliki informasi Omset (sangat krusial)
df = df[df['Omset per-Tahun'] != '-'].dropna(subset=['Omset per-Tahun'])
df = df.reset_index(drop=True)

### 3. Ekstraksi Fitur Numerik (Feature Engineering)
Karena K-Means hanya menerima data numerik, kita harus mengekstrak informasi dari kolom teks/kategorikal menjadi skala numerik.

In [5]:
le = LabelEncoder()

# --- A. Fitur Demografi & Tenaga Kerja ---
df['Usia Numeric'] = df['Usia']
df['Total Pekerja Numeric'] = df['Laki-laki'] + df['Perempuan']
df['Jenis Kelamin Numeric'] = le.fit_transform(df['Jenis Kelamin'].astype(str))
df['Pendidikan Numeric'] = le.fit_transform(df['Pendidikan Terakhir'].astype(str))
df['Rerata Usia Pekerja Numeric'] = le.fit_transform(df['Rerata Usia Pekerja'].astype(str))

# --- B. Fitur Finansial & Aset ---
def extract_omset(val):
    val = str(val)
    if 'Kurang dari 10 juta' in val: return 5.0
    elif '10 juta s/d 25 juta' in val: return 17.5
    elif '25 juta s/d 40 juta' in val: return 32.5
    elif '40 juta s/d 55 juta' in val: return 47.5
    elif '55 juta s/d 70 juta' in val: return 62.5
    elif '70 juta s/d 85 juta' in val: return 77.5
    elif '85 juta s/d 100 juta' in val: return 92.5
    elif '100 juta s/d 120 juta' in val: return 110.0
    elif '120 juta s/d 150 juta' in val: return 135.0
    elif 'Lebih dari 150 juta' in val: return 150.0
    return 5.0

df['Omset Numeric'] = df['Omset per-Tahun'].apply(extract_omset)

# Bantuan Pemerintah & Pinjaman (1 jika dapat, 0 jika tidak/-)
df['Bantuan Pemerintah Numeric'] = df['Modal Bantuan Pemerintah'].apply(lambda x: 0 if pd.isna(x) or str(x).strip() == '-' else 1)
df['Pinjaman Bank Numeric'] = df['Pinjaman Kredit Usaha Rakyat'].apply(lambda x: 0 if pd.isna(x) or str(x).strip() == '-' else 1)
df['Kepemilkan Tanah Numeric'] = le.fit_transform(df['Status Kepemilkan Tanah/Bangunan'].astype(str))

# --- C. Fitur Digitalisasi ---
# Menghitung berapa banyak platform yang digunakan (WA, IG, FB, dll)
df['Skor Digitalisasi'] = df['Sarana Media Elektronik'].apply(lambda x: 0 if pd.isna(x) or str(x).strip() == '-' else len(str(x).split(',')))

# --- D. Fitur Geografis & Pemasaran ---
df['Tujuan Pemasaran Numeric'] = le.fit_transform(df['Tujuan Pemasaran'].astype(str))
df['Kecamatan Numeric'] = le.fit_transform(df['Kecamatan'].astype(str))

# --- E. Fitur Profil Usaha ---
df['Sektor Numeric'] = le.fit_transform(df['Sektor Usaha'].astype(str))
df['Bidang Usaha Numeric'] = le.fit_transform(df['Bidang Usaha'].astype(str))
df['Ekspor Numeric'] = df['Produk Komoditas Ekspor'].apply(lambda x: 1 if str(x).strip().lower() == 'ya' else 0)


### 4. Simpan Data Bersih
Menyimpan ke dalam file baru agar tidak merusak data asli, dan notebook lain bisa langsung load data numerik ini.

In [6]:
clean_file_path = '../data/Data_UMKM_Clean.xlsx'
df.to_excel(clean_file_path, index=False)
print(f"Data bersih berhasil disimpan ke: {clean_file_path}")
print(f"Total kolom saat ini: {df.shape[1]}")

Data bersih berhasil disimpan ke: ../data/Data_UMKM_Clean.xlsx
Total kolom saat ini: 46
